In [ ]:
import os
import pandas as pd
import mysql.connector
from pymongo import MongoClient

mysql_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=os.getenv("ROUTESENSE_MYSQL_PASSWORD"),
    database="delivery_db"
)

print("MySQL connected")
print("Database:", mysql_conn.database)

mongo_client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

mongo_client.admin.command("ping")

mongo_db = mongo_client["routesense"]

print("MongoDB connected")
print("Database:", mongo_db.name)

In [ ]:
query = """
SELECT
    o.order_id,
    o.olist_order_id,
    o.customer_id,
    o.route_id,
    o.order_date,
    o.promised_date,
    o.delivery_date,
    o.status AS order_status,
    o.amount,
    o.freight_value,
    o.zone AS order_zone,

    c.customer_unique_id,
    c.city,
    c.zone AS customer_zone,
    c.region,
    c.pincode,

    r.route_name,
    r.origin_hub,
    r.distance_km,
    r.driver_id,
    r.vehicle_id,

    d.name AS driver_name,
    d.experience_years,

    v.vehicle_type,
    v.capacity_kg,

    da.attempt_id,
    da.attempt_no,
    da.attempt_time,
    da.status AS attempt_status,
    da.failure_reason,
    da.delay_minutes

FROM delivery_attempts da

JOIN orders o
    ON da.order_id = o.order_id

JOIN customers c
    ON o.customer_id = c.customer_id

JOIN routes r
    ON o.route_id = r.route_id

JOIN drivers d
    ON r.driver_id = d.driver_id

JOIN vehicles v
    ON r.vehicle_id = v.vehicle_id
"""

integrated_df = pd.read_sql(query, mysql_conn)

print("\nSQL JOIN successful")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

print("\nUnique values:")
print("Orders   :", integrated_df["order_id"].nunique())
print("Customers:", integrated_df["customer_id"].nunique())
print("Routes   :", integrated_df["route_id"].nunique())
print("Drivers  :", integrated_df["driver_id"].nunique())
print("Vehicles :", integrated_df["vehicle_id"].nunique())

print("\nSample:")
print(integrated_df.head())

In [ ]:
key_columns = [
    "attempt_id",
    "order_id",
    "customer_id",
    "route_id",
    "driver_id",
    "vehicle_id"
]

for col in key_columns:
    print(
        f"{col:20} : "
        f"{integrated_df[col].isna().sum()}"
    )

print("\nDuplicate attempt IDs:")
print(
    integrated_df["attempt_id"].duplicated().sum()
)

print("\nAttempt status:")
print(
    integrated_df["attempt_status"].value_counts()
)

print("\nFailure reasons:")
print(
    integrated_df["failure_reason"]
    .value_counts(dropna=False)
)

print("\nRoutes:")
print(
    integrated_df["route_id"]
    .nunique()
)

print("\nDrivers:")
print(
    integrated_df["driver_id"]
    .nunique()
)

print("\nVehicles:")
print(
    integrated_df["vehicle_id"]
    .nunique()
)

In [ ]:
mongo_events = []

for _, row in integrated_df.iterrows():

    attempt_time = row["attempt_time"]

    if pd.notna(attempt_time):
        attempt_time = pd.Timestamp(
            attempt_time
        ).to_pydatetime()
    else:
        attempt_time = None

    order_date = (
        pd.Timestamp(row["order_date"]).date()
        if pd.notna(row["order_date"])
        else None
    )

    promised_date = (
        pd.Timestamp(row["promised_date"]).date()
        if pd.notna(row["promised_date"])
        else None
    )

    delivery_date = (
        pd.Timestamp(row["delivery_date"]).date()
        if pd.notna(row["delivery_date"])
        else None
    )

    delay = row["delay_minutes"]

    if pd.isna(delay):
        delay = 0
    else:
        delay = int(delay)

    if delay == 0:
        delay_bucket = "NO_DELAY"
    elif delay <= 60:
        delay_bucket = "0_60_MIN"
    elif delay <= 120:
        delay_bucket = "61_120_MIN"
    else:
        delay_bucket = "OVER_120_MIN"

    document = {
        "event_id": str(row["attempt_id"]),
        "event_type": (
            "DELIVERY_FAILED"
            if row["attempt_status"] == "Failed"
            else "DELIVERY_COMPLETED"
        ),
        "timestamp": attempt_time,

        "order": {
            "order_id": str(row["order_id"]),
            "olist_order_id": str(row["olist_order_id"]),
            "order_date": order_date,
            "promised_date": promised_date,
            "delivery_date": delivery_date,
            "status": str(row["order_status"]),
            "amount": (
                float(row["amount"])
                if pd.notna(row["amount"])
                else None
            ),
            "freight_value": (
                float(row["freight_value"])
                if pd.notna(row["freight_value"])
                else None
            )
        },

        "customer": {
            "customer_id": str(row["customer_id"]),
            "customer_unique_id": str(
                row["customer_unique_id"]
            ),
            "city": row["city"],
            "zone": row["customer_zone"],
            "region": row["region"],
            "pincode": row["pincode"]
        },

        "route": {
            "route_id": str(row["route_id"]),
            "route_name": row["route_name"],
            "origin_hub": row["origin_hub"],
            "zone": row["order_zone"],
            "distance_km": (
                float(row["distance_km"])
                if pd.notna(row["distance_km"])
                else None
            )
        },

        "driver": {
            "driver_id": str(row["driver_id"]),
            "name": row["driver_name"],
            "experience_years": (
                int(row["experience_years"])
                if pd.notna(row["experience_years"])
                else None
            )
        },

        "vehicle": {
            "vehicle_id": str(row["vehicle_id"]),
            "type": row["vehicle_type"],
            "capacity_kg": (
                int(row["capacity_kg"])
                if pd.notna(row["capacity_kg"])
                else None
            )
        },

        "attempt": {
            "attempt_no": int(row["attempt_no"]),
            "status": str(row["attempt_status"]),
            "failure_reason": (
                row["failure_reason"]
                if pd.notna(row["failure_reason"])
                else None
            ),
            "delay_minutes": delay,
            "delay_bucket": delay_bucket
        }
    }

    mongo_events.append(document)

print("MongoDB documents prepared")
print("Documents:", len(mongo_events))

print("\nSample document:")
print(mongo_events[0])

In [ ]:
client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

client.admin.command("ping")

mongo_db = client["routesense"]
collection = mongo_db["delivery_events"]

print("MongoDB connected")
print("Database:", mongo_db.name)
print("Collection:", collection.name)

In [ ]:
def to_datetime(value):
    if pd.isna(value):
        return None
    return pd.Timestamp(value).to_pydatetime()

mongo_events = []

for _, row in integrated_df.iterrows():

    document = {
        "event_id": str(row["attempt_id"]),

        "event_type": (
            "DELIVERY_FAILED"
            if row["attempt_status"] == "Failed"
            else "DELIVERY_COMPLETED"
        ),

        "timestamp": to_datetime(row["attempt_time"]),

        "order": {
            "order_id": str(row["order_id"]),
            "olist_order_id": str(row["olist_order_id"]),
            "order_date": to_datetime(row["order_date"]),
            "promised_date": to_datetime(row["promised_date"]),
            "delivery_date": to_datetime(row["delivery_date"]),
            "status": str(row["order_status"]),
            "amount": (
                float(row["amount"])
                if pd.notna(row["amount"])
                else None
            ),
            "freight_value": (
                float(row["freight_value"])
                if pd.notna(row["freight_value"])
                else None
            )
        },

        "customer": {
            "customer_id": str(row["customer_id"]),
            "customer_unique_id": str(row["customer_unique_id"]),
            "city": row["city"],
            "zone": row["customer_zone"],
            "region": row["region"],
            "pincode": row["pincode"]
        },

        "route": {
            "route_id": str(row["route_id"]),
            "route_name": row["route_name"],
            "origin_hub": row["origin_hub"],
            "zone": row["order_zone"],
            "distance_km": (
                float(row["distance_km"])
                if pd.notna(row["distance_km"])
                else None
            )
        },

        "driver": {
            "driver_id": str(row["driver_id"]),
            "name": row["driver_name"],
            "experience_years": (
                int(row["experience_years"])
                if pd.notna(row["experience_years"])
                else None
            )
        },

        "vehicle": {
            "vehicle_id": str(row["vehicle_id"]),
            "type": row["vehicle_type"],
            "capacity_kg": (
                int(row["capacity_kg"])
                if pd.notna(row["capacity_kg"])
                else None
            )
        },

        "attempt": {
            "attempt_no": int(row["attempt_no"]),
            "status": str(row["attempt_status"]),
            "failure_reason": (
                row["failure_reason"]
                if pd.notna(row["failure_reason"])
                else None
            ),
            "delay_minutes": (
                int(row["delay_minutes"])
                if pd.notna(row["delay_minutes"])
                else 0
            )
        }
    }

    mongo_events.append(document)

print("MongoDB documents prepared:", len(mongo_events))
print("\nSample document:")
print(mongo_events[0])

In [ ]:
print(type(mongo_events[0]["order"]["order_date"]))
print(type(mongo_events[0]["order"]["promised_date"]))
print(type(mongo_events[0]["order"]["delivery_date"]))
print(type(mongo_events[0]["timestamp"]))

In [ ]:
collection.delete_many({})

batch_size = 5000

for start in range(0, len(mongo_events), batch_size):

    batch = mongo_events[start:start + batch_size]

    collection.insert_many(batch)

    inserted = min(
        start + batch_size,
        len(mongo_events)
    )

    print(
        f"Inserted {inserted:,} / "
        f"{len(mongo_events):,}"
    )

print("\nMongoDB insertion complete")
print("Final documents:", collection.count_documents({}))

In [ ]:
print("Total documents:", collection.count_documents({}))

print(
    "Unique event IDs:",
    len(collection.distinct("event_id"))
)

print(
    "Failed events:",
    collection.count_documents({
        "attempt.status": "Failed"
    })
)

print(
    "Delivered events:",
    collection.count_documents({
        "attempt.status": "Delivered"
    })
)

print("\nSample document:")
print(
    collection.find_one(
        {},
        {"_id": 0}
    )
)

In [ ]:
pipeline = [
    {
        "$match": {
            "attempt.status": "Failed"
        }
    },
    {
        "$group": {
            "_id": "$attempt.failure_reason",
            "failure_count": {
                "$sum": 1
            },
            "average_delay": {
                "$avg": "$attempt.delay_minutes"
            },
            "maximum_delay": {
                "$max": "$attempt.delay_minutes"
            },
            "total_delay": {
                "$sum": "$attempt.delay_minutes"
            }
        }
    },
    {
        "$sort": {
            "failure_count": -1
        }
    }
]

failure_results = list(
    collection.aggregate(pipeline)
)

for result in failure_results:
    print(
        f"Reason: {result['_id']} | "
        f"Failures: {result['failure_count']:,} | "
        f"Avg Delay: {result['average_delay']:.2f} min | "
        f"Max Delay: {result['maximum_delay']:,} min"
    )

print("\nTotal failed events:", sum(
    r["failure_count"] for r in failure_results
))

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": {
                "route_id": "$route.route_id",
                "order_id": "$order.order_id"
            },
            "failed": {
                "$max": {
                    "$cond": [
                        {
                            "$eq": [
                                "$attempt.status",
                                "Failed"
                            ]
                        },
                        1,
                        0
                    ]
                },
            },
            "total_delay": {
                "$sum": "$attempt.delay_minutes"
            }
        }
    },
    {
        "$group": {
            "_id": "$_id.route_id",
            "total_orders": {
                "$sum": 1
            },
            "failed_orders": {
                "$sum": "$failed"
            },
            "total_delay": {
                "$sum": "$total_delay"
            }
        }
    },
    {
        "$match": {
            "total_orders": {
                "$gte": 100
            }
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_orders": 1,
            "failed_orders": 1,
            "failure_rate": {
                "$multiply": [
                    {
                        "$divide": [
                            "$failed_orders",
                            "$total_orders"
                        ]
                    },
                    100
                ]
            },
            "average_delay_per_order": {
                "$divide": [
                    "$total_delay",
                    "$total_orders"
                ]
            }
        }
    },
    {
        "$sort": {
            "failure_rate": -1
        }
    }
]

route_results = list(
    collection.aggregate(pipeline)
)

for result in route_results:
    print(
        f"Route: {result['_id']} | "
        f"Orders: {result['total_orders']:,} | "
        f"Failed Orders: {result['failed_orders']:,} | "
        f"Failure Rate: {result['failure_rate']:.2f}% | "
        f"Avg Delay/Order: {result['average_delay_per_order']:.2f} min"
    )

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": None,
            "minimum_delay": {
                "$min": "$attempt.delay_minutes"
            },
            "maximum_delay": {
                "$max": "$attempt.delay_minutes"
            },
            "average_delay": {
                "$avg": "$attempt.delay_minutes"
            },
            "delays_over_24h": {
                "$sum": {
                    "$cond": [
                        {
                            "$gt": [
                                "$attempt.delay_minutes",
                                1440
                            ]
                        },
                        1,
                        0
                    ]
                }
            },
            "failed_events": {
                "$sum": {
                    "$cond": [
                        {
                            "$eq": [
                                "$attempt.status",
                                "Failed"
                            ]
                        },
                        1,
                        0
                    ]
                }
            }
        }
    }
]

result = list(
    collection.aggregate(pipeline)
)[0]

print("Minimum delay:", result["minimum_delay"], "min")
print("Maximum delay:", result["maximum_delay"], "min")
print("Average delay:", round(result["average_delay"], 2), "min")
print("Delays > 24 hours:", result["delays_over_24h"])
print("Failed events:", result["failed_events"])

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": {
                "route_id": "$route.route_id",
                "order_id": "$order.order_id"
            },
            "attempt_count": {
                "$sum": 1
            },
            "failed": {
                "$max": {
                    "$cond": [
                        {
                            "$eq": [
                                "$attempt.status",
                                "Failed"
                            ]
                        },
                        1,
                        0
                    ]
                }
            },
            "failed_delays": {
                "$push": {
                    "$cond": [
                        {
                            "$and": [
                                {
                                    "$eq": [
                                        "$attempt.status",
                                        "Failed"
                                    ]
                                },
                                {
                                    "$lte": [
                                        "$attempt.delay_minutes",
                                        1440
                                    ]
                                }
                            ]
                        },
                        "$attempt.delay_minutes",
                        None
                    ]
                }
            }
        }
    },
    {
        "$group": {
            "_id": "$_id.route_id",
            "total_orders": {
                "$sum": 1
            },
            "failed_orders": {
                "$sum": "$failed"
            },
            "all_failed_delays": {
                "$push": "$failed_delays"
            }
        }
    },
    {
        "$match": {
            "total_orders": {
                "$gte": 100
            }
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_orders": 1,
            "failed_orders": 1,
            "failure_rate": {
                "$multiply": [
                    {
                        "$divide": [
                            "$failed_orders",
                            "$total_orders"
                        ]
                    },
                    100
                ]
            }
        }
    },
    {
        "$sort": {
            "failure_rate": -1
        }
    }
]

route_results = list(
    collection.aggregate(pipeline)
)

for result in route_results:
    print(
        f"Route: {result['_id']} | "
        f"Orders: {result['total_orders']:,} | "
        f"Failed Orders: {result['failed_orders']:,} | "
        f"Failure Rate: {result['failure_rate']:.2f}%"
    )

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": {
                "zone": "$route.zone",
                "order_id": "$order.order_id"
            },
            "failed": {
                "$max": {
                    "$cond": [
                        {
                            "$eq": [
                                "$attempt.status",
                                "Failed"
                            ]
                        },
                        1,
                        0
                    ]
                }
            },
            "delay": {
                "$max": "$attempt.delay_minutes"
            }
        }
    },
    {
        "$group": {
            "_id": "$_id.zone",
            "total_orders": {
                "$sum": 1
            },
            "failed_orders": {
                "$sum": "$failed"
            },
            "total_delay": {
                "$sum": "$delay"
            }
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_orders": 1,
            "failed_orders": 1,
            "failure_rate": {
                "$multiply": [
                    {
                        "$divide": [
                            "$failed_orders",
                            "$total_orders"
                        ]
                    },
                    100
                ]
            },
            "average_delay_per_order": {
                "$divide": [
                    "$total_delay",
                    "$total_orders"
                ]
            }
        }
    },
    {
        "$sort": {
            "failure_rate": -1
        }
    }
]

zone_results = list(
    collection.aggregate(pipeline)
)

for result in zone_results:
    print(
        f"Zone: {result['_id']} | "
        f"Orders: {result['total_orders']:,} | "
        f"Failed Orders: {result['failed_orders']:,} | "
        f"Failure Rate: {result['failure_rate']:.2f}% | "
        f"Avg Delay/Order: {result['average_delay_per_order']:.2f} min"
    )

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": {
                "$hour": "$timestamp"
            },
            "total_attempts": {
                "$sum": 1
            },
            "failed_attempts": {
                "$sum": {
                    "$cond": [
                        {
                            "$eq": [
                                "$attempt.status",
                                "Failed"
                            ]
                        },
                        1,
                        0
                    ]
                }
            },
            "average_delay": {
                "$avg": "$attempt.delay_minutes"
            }
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_attempts": 1,
            "failed_attempts": 1,
            "average_delay": 1,
            "failure_rate": {
                "$multiply": [
                    {
                        "$divide": [
                            "$failed_attempts",
                            "$total_attempts"
                        ]
                    },
                    100
                ]
            }
        }
    },
    {
        "$sort": {
            "failure_rate": -1
        }
    }
]

hour_results = list(
    collection.aggregate(pipeline)
)

for result in hour_results:
    print(
        f"Hour: {result['_id']:02d}:00 | "
        f"Attempts: {result['total_attempts']:,} | "
        f"Failures: {result['failed_attempts']:,} | "
        f"Failure Rate: {result['failure_rate']:.2f}% | "
        f"Avg Delay: {result['average_delay']:.2f} min"
    )

In [ ]:
pipeline = [
    {
        "$match": {
            "attempt.status": "Failed"
        }
    },
    {
        "$group": {
            "_id": "$order.order_id",
            "failed_attempts": {
                "$sum": 1
            },
            "failure_reasons": {
                "$addToSet": "$attempt.failure_reason"
            }
        }
    },
    {
        "$match": {
            "failed_attempts": {
                "$gt": 1
            }
        }
    },
    {
        "$sort": {
            "failed_attempts": -1
        }
    }
]

repeat_results = list(
    collection.aggregate(pipeline)
)

print(
    "Orders with multiple failed attempts:",
    len(repeat_results)
)

print("\nTop repeated-failure orders:")

for result in repeat_results[:20]:
    print(
        f"Order: {result['_id']} | "
        f"Failed Attempts: {result['failed_attempts']} | "
        f"Reasons: {', '.join(result['failure_reasons'])}"
    )

In [ ]:
print("REPEATED FAILURE ANALYSIS BY ROUTE")
pipeline = [
    {
        "$group": {
            "_id": {
                "route_id": "$route.route_id",
                "order_id": "$order.order_id"
            },
            "failed_attempts": {
                "$sum": {
                    "$cond": [
                        {
                            "$eq": [
                                "$attempt.status",
                                "Failed"
                            ]
                        },
                        1,
                        0
                    ]
                }
            }
        }
    },
    {
        "$group": {
            "_id": "$_id.route_id",
            "total_orders": {
                "$sum": 1
            },
            "repeat_failure_orders": {
                "$sum": {
                    "$cond": [
                        {
                            "$gt": [
                                "$failed_attempts",
                                1
                            ]
                        },
                        1,
                        0
                    ]
                }
            }
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_orders": 1,
            "repeat_failure_orders": 1,
            "repeat_failure_rate": {
                "$multiply": [
                    {
                        "$divide": [
                            "$repeat_failure_orders",
                            "$total_orders"
                        ]
                    },
                    100
                ]
            }
        }
    },
    {
        "$sort": {
            "repeat_failure_rate": -1
        }
    }
]

repeat_route_results = list(
    collection.aggregate(pipeline)
)

for result in repeat_route_results:
    print(
        f"Route: {result['_id']} | "
        f"Orders: {result['total_orders']:,} | "
        f"Repeat-Failure Orders: {result['repeat_failure_orders']:,} | "
        f"Repeat-Failure Rate: {result['repeat_failure_rate']:.2f}%"
    )

In [ ]:
print("ROUTESENSE HOTSPOT ENGINE")

pipeline = [
    {
        "$group": {
            "_id": {
                "route_id": "$route.route_id",
                "order_id": "$order.order_id"
            },
            "failed": {
                "$max": {
                    "$cond": [
                        {"$eq": ["$attempt.status", "Failed"]},
                        1,
                        0
                    ]
                },
            },
            "failed_attempts": {
                "$sum": {
                    "$cond": [
                        {"$eq": ["$attempt.status", "Failed"]},
                        1,
                        0
                    ]
                }
            }
        }
    },
    {
        "$group": {
            "_id": "$_id.route_id",
            "total_orders": {"$sum": 1},
            "failed_orders": {"$sum": "$failed"},
            "repeat_failure_orders": {
                "$sum": {
                    "$cond": [
                        {"$gt": ["$failed_attempts", 1]},
                        1,
                        0
                    ]
                }
            }
        }
    },
    {
        "$match": {
            "total_orders": {"$gte": 100}
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_orders": 1,
            "failed_orders": 1,
            "repeat_failure_orders": 1,
            "failure_rate": {
                "$multiply": [
                    {"$divide": ["$failed_orders", "$total_orders"]},
                    100
                ]
            },
            "repeat_failure_rate": {
                "$multiply": [
                    {
                        "$divide": [
                            "$repeat_failure_orders",
                            "$total_orders"
                        ]
                    },
                    100
                ]
            }
        }
    },
    {
        "$setWindowFields": {
            "partitionBy": None,
            "sortBy": {"failure_rate": 1},
            "output": {
                "max_failure_rate": {
                    "$max": "$failure_rate",
                    "window": {
                        "documents": ["unbounded", "unbounded"]
                    }
                },
                "max_repeat_failure_rate": {
                    "$max": "$repeat_failure_rate",
                    "window": {
                        "documents": ["unbounded", "unbounded"]
                    }
                }
            }
        }
    },
    {
        "$project": {
            "_id": 1,
            "total_orders": 1,
            "failed_orders": 1,
            "repeat_failure_orders": 1,
            "failure_rate": 1,
            "repeat_failure_rate": 1,
            "hotspot_score": {
                "$add": [
                    {
                        "$multiply": [
                            {
                                "$divide": [
                                    "$failure_rate",
                                    "$max_failure_rate"
                                ]
                            },
                            60
                        ]
                    },
                    {
                        "$multiply": [
                            {
                                "$divide": [
                                    "$repeat_failure_rate",
                                    "$max_repeat_failure_rate"
                                ]
                            },
                            30
                        ]
                    },
                    {
                        "$multiply": [
                            {
                                "$divide": [
                                    "$total_orders",
                                    41119
                                ]
                            },
                            10
                        ]
                    }
                ]
            }
        }
    },
    {
        "$sort": {
            "hotspot_score": -1
        }
    }
]

hotspot_results = list(
    collection.aggregate(pipeline)
)

for result in hotspot_results:
    print(
        f"Route: {result['_id']} | "
        f"Orders: {result['total_orders']:,} | "
        f"Failure Rate: {result['failure_rate']:.2f}% | "
        f"Repeat Failure Rate: {result['repeat_failure_rate']:.2f}% | "
        f"Hotspot Score: {result['hotspot_score']:.2f}"
    )

In [ ]:
print("FINAL ROUTESENSE HOTSPOT DATASET")

hotspots_df = pd.DataFrame(hotspot_results)

route_info = (
    integrated_df[
        [
            "route_id",
            "route_name",
            "order_zone",
            "origin_hub",
            "distance_km",
            "driver_id",
            "vehicle_id"
        ]
    ]
    .drop_duplicates("route_id")
)

hotspots_df = hotspots_df.merge(
    route_info,
    left_on="_id",
    right_on="route_id",
    how="left"
)

hotspots_df = hotspots_df.rename(
    columns={
        "_id": "route_id"
    }
)

hotspots_df = hotspots_df[
    [
        "route_id",
        "route_name",
        "order_zone",
        "origin_hub",
        "distance_km",
        "driver_id",
        "vehicle_id",
        "total_orders",
        "failed_orders",
        "repeat_failure_orders",
        "failure_rate",
        "repeat_failure_rate",
        "hotspot_score"
    ]
]

print("Rows:", len(hotspots_df))
print("Columns:", len(hotspots_df.columns))

print("\nFinal hotspot table:")
print(hotspots_df.head(10))

In [ ]:
from pathlib import Path

analytics_dir = Path(
    r"<PROJECT_ROOT>\data\analytics"
)

analytics_dir.mkdir(
    parents=True,
    exist_ok=True
)

failure_df = pd.DataFrame(failure_results)

zone_df = pd.DataFrame(zone_results)

hour_df = pd.DataFrame(hour_results)

repeat_route_df = pd.DataFrame(
    repeat_route_results
)

hotspots_df.to_csv(
    analytics_dir / "route_hotspots.csv",
    index=False
)

failure_df.to_csv(
    analytics_dir / "failure_reason_analysis.csv",
    index=False
)

zone_df.to_csv(
    analytics_dir / "zone_analysis.csv",
    index=False
)

hour_df.to_csv(
    analytics_dir / "hourly_analysis.csv",
    index=False
)

repeat_route_df.to_csv(
    analytics_dir / "repeat_failure_routes.csv",
    index=False
)

for file in sorted(analytics_dir.glob("*.csv")):
    print(
        f"{file.name:35} "
        f"{file.stat().st_size / 1024:.1f} KB"
    )

In [ ]:
checks = {}

checks["MySQL connection"] = mysql_conn.is_connected()

mongo_client.admin.command("ping")
checks["MongoDB connection"] = True

checks["Integrated rows"] = (
    len(integrated_df) == 117916
)

checks["MongoDB document count"] = (
    collection.count_documents({}) == 117916
)

checks["Unique event IDs"] = (
    len(collection.distinct("event_id")) == 117916
)

checks["Missing order IDs"] = (
    integrated_df["order_id"].isna().sum() == 0
)

checks["Missing route IDs"] = (
    integrated_df["route_id"].isna().sum() == 0
)

checks["Duplicate event IDs"] = (
    integrated_df["attempt_id"].duplicated().sum() == 0
)

checks["Exported hotspot file"] = (
    (analytics_dir / "route_hotspots.csv").exists()
)

for name, result in checks.items():
    print(
        f"{'PASS' if result else 'FAIL'} | {name}"
    )

print("\nValidation complete.")

In [ ]:
from pathlib import Path

ROOT = Path(
    r"<PROJECT_ROOT>"
)

SRC = ROOT / "src"
SRC.mkdir(parents=True, exist_ok=True)

files = {}

files["__init__.py"] = ""

files["database.py"] = r'''
import os
from pathlib import Path

import mysql.connector
import pandas as pd
from dotenv import load_dotenv

ROOT = Path(
    r"<PROJECT_ROOT>"
)

load_dotenv(ROOT / ".env")

QUERY = """
SELECT
    o.order_id,
    o.olist_order_id,
    o.customer_id,
    o.route_id,
    o.order_date,
    o.promised_date,
    o.delivery_date,
    o.status AS order_status,
    o.amount,
    o.freight_value,
    o.zone AS order_zone,

    c.customer_unique_id,
    c.city,
    c.zone AS customer_zone,
    c.region,
    c.pincode,

    r.route_name,
    r.origin_hub,
    r.distance_km,
    r.driver_id,
    r.vehicle_id,

    d.name AS driver_name,
    d.experience_years,

    v.vehicle_type,
    v.capacity_kg,

    da.attempt_id,
    da.attempt_no,
    da.attempt_time,
    da.status AS attempt_status,
    da.failure_reason,
    da.delay_minutes

FROM delivery_attempts da

JOIN orders o
    ON da.order_id = o.order_id

JOIN customers c
    ON o.customer_id = c.customer_id

JOIN routes r
    ON o.route_id = r.route_id

JOIN drivers d
    ON r.driver_id = d.driver_id

JOIN vehicles v
    ON r.vehicle_id = v.vehicle_id
"""


def get_mysql_connection():
    password = os.getenv("ROUTESENSE_MYSQL_PASSWORD")

    if not password:
        raise RuntimeError(
            "ROUTESENSE_MYSQL_PASSWORD is not configured."
        )

    return mysql.connector.connect(
        host="localhost",
        user="root",
        password=password,
        database="delivery_db"
    )


def load_integrated_data():
    conn = get_mysql_connection()

    try:
        return pd.read_sql(QUERY, conn)

    finally:
        conn.close()
'''

files["mongodb.py"] = r'''
from pymongo import MongoClient
from pymongo.errors import PyMongoError


def get_collection():
    client = MongoClient(
        "mongodb://localhost:27017/",
        serverSelectionTimeoutMS=5000
    )

    client.admin.command("ping")

    return client["routesense"]["delivery_events"]


def get_kpis(collection):
    total = collection.count_documents({})

    failed = collection.count_documents({
        "attempt.status": "Failed"
    })

    delivered = collection.count_documents({
        "attempt.status": "Delivered"
    })

    failure_rate = (
        failed / total * 100
        if total else 0
    )

    return {
        "total_attempts": total,
        "failed_attempts": failed,
        "delivered_attempts": delivered,
        "failure_rate": failure_rate
    }


def get_failure_summary(collection):
    pipeline = [
        {
            "$match": {
                "attempt.status": "Failed"
            }
        },
        {
            "$group": {
                "_id": "$attempt.failure_reason",
                "failure_count": {"$sum": 1},
                "average_delay": {
                    "$avg": "$attempt.delay_minutes"
                }
            }
        },
        {
            "$sort": {
                "failure_count": -1
            }
        }
    ]

    return list(
        collection.aggregate(pipeline)
    )


def get_hourly_summary(collection):
    pipeline = [
        {
            "$group": {
                "_id": {
                    "$hour": "$timestamp"
                },
                "total_attempts": {
                    "$sum": 1
                },
                "failed_attempts": {
                    "$sum": {
                        "$cond": [
                            {
                                "$eq": [
                                    "$attempt.status",
                                    "Failed"
                                ]
                            },
                            1,
                            0
                        ]
                    }
                }
            }
        },
        {
            "$project": {
                "_id": 1,
                "total_attempts": 1,
                "failed_attempts": 1,
                "failure_rate": {
                    "$multiply": [
                        {
                            "$divide": [
                                "$failed_attempts",
                                "$total_attempts"
                            ]
                        },
                        100
                    ]
                }
            }
        },
        {
            "$sort": {
                "_id": 1
            }
        }
    ]

    return list(
        collection.aggregate(pipeline)
    )


def get_route_hotspots(collection):
    pipeline = [
        {
            "$group": {
                "_id": {
                    "route_id": "$route.route_id",
                    "order_id": "$order.order_id"
                },
                "failed": {
                    "$max": {
                        "$cond": [
                            {
                                "$eq": [
                                    "$attempt.status",
                                    "Failed"
                                ]
                            },
                            1,
                            0
                        ]
                    }
                },
                "failed_attempts": {
                    "$sum": {
                        "$cond": [
                            {
                                "$eq": [
                                    "$attempt.status",
                                    "Failed"
                                ]
                            },
                            1,
                            0
                        ]
                    }
                }
            }
        },
        {
            "$group": {
                "_id": "$_id.route_id",
                "total_orders": {
                    "$sum": 1
                },
                "failed_orders": {
                    "$sum": "$failed"
                },
                "repeat_failure_orders": {
                    "$sum": {
                        "$cond": [
                            {
                                "$gt": [
                                    "$failed_attempts",
                                    1
                                ]
                            },
                            1,
                            0
                        ]
                    }
                }
            }
        },
        {
            "$match": {
                "total_orders": {
                    "$gte": 100
                }
            }
        },
        {
            "$project": {
                "_id": 1,
                "total_orders": 1,
                "failed_orders": 1,
                "repeat_failure_orders": 1,
                "failure_rate": {
                    "$multiply": [
                        {
                            "$divide": [
                                "$failed_orders",
                                "$total_orders"
                            ]
                        },
                        100
                    ]
                },
                "repeat_failure_rate": {
                    "$multiply": [
                        {
                            "$divide": [
                                "$repeat_failure_orders",
                                "$total_orders"
                            ]
                        },
                        100
                    ]
                }
            }
        },
        {
            "$setWindowFields": {
                "partitionBy": None,
                "output": {
                    "max_failure_rate": {
                        "$max": "$failure_rate",
                        "window": {
                            "documents": [
                                "unbounded",
                                "unbounded"
                            ]
                        }
                    },
                    "max_repeat_failure_rate": {
                        "$max": "$repeat_failure_rate",
                        "window": {
                            "documents": [
                                "unbounded",
                                "unbounded"
                            ]
                        }
                    },
                    "max_total_orders": {
                        "$max": "$total_orders",
                        "window": {
                            "documents": [
                                "unbounded",
                                "unbounded"
                            ]
                        }
                    }
                }
            }
        },
        {
            "$project": {
                "_id": 1,
                "total_orders": 1,
                "failed_orders": 1,
                "repeat_failure_orders": 1,
                "failure_rate": 1,
                "repeat_failure_rate": 1,
                "hotspot_score": {
                    "$add": [
                        {
                            "$multiply": [
                                {
                                    "$divide": [
                                        "$failure_rate",
                                        "$max_failure_rate"
                                    ]
                                },
                                60
                            ]
                        },
                        {
                            "$multiply": [
                                {
                                    "$divide": [
                                        "$repeat_failure_rate",
                                        "$max_repeat_failure_rate"
                                    ]
                                },
                                30
                            ]
                        },
                        {
                            "$multiply": [
                                {
                                    "$divide": [
                                        "$total_orders",
                                        "$max_total_orders"
                                    ]
                                },
                                10
                            ]
                        }
                    ]
                }
            }
        },
        {
            "$sort": {
                "hotspot_score": -1
            }
        }
    ]

    return list(
        collection.aggregate(pipeline)
    )
'''

files["analytics.py"] = r'''
import pandas as pd


class RouteSenseAnalytics:

    def __init__(self, integrated_df):
        self.df = integrated_df.copy()

    def failure_summary(self):
        failed = self.df[
            self.df["attempt_status"] == "Failed"
        ]

        return (
            failed
            .groupby("failure_reason")
            .agg(
                failures=("attempt_id", "count"),
                average_delay=("delay_minutes", "mean"),
                maximum_delay=("delay_minutes", "max")
            )
            .sort_values(
                "failures",
                ascending=False
            )
            .reset_index()
        )

    def route_summary(self):
        result = (
            self.df
            .groupby("route_id")
            .agg(
                total_orders=(
                    "order_id",
                    "nunique"
                ),
                failed_attempts=(
                    "attempt_status",
                    lambda x: (
                        x == "Failed"
                    ).sum()
                )
            )
            .reset_index()
        )

        failed_orders = (
            self.df[
                self.df["attempt_status"] == "Failed"
            ]
            .groupby("route_id")["order_id"]
            .nunique()
            .rename("failed_orders")
        )

        result = result.merge(
            failed_orders,
            on="route_id",
            how="left"
        )

        result["failed_orders"] = (
            result["failed_orders"]
            .fillna(0)
            .astype(int)
        )

        result["failure_rate"] = (
            result["failed_orders"]
            / result["total_orders"]
            * 100
        )

        return result.sort_values(
            "failure_rate",
            ascending=False
        )

    def zone_summary(self):
        result = (
            self.df
            .groupby("order_zone")
            .agg(
                total_orders=(
                    "order_id",
                    "nunique"
                ),
                failed_orders=(
                    "attempt_status",
                    lambda x: (
                        x == "Failed"
                    ).sum()
                )
            )
            .reset_index()
        )

        result["failure_rate"] = (
            result["failed_orders"]
            / result["total_orders"]
            * 100
        )

        return result.sort_values(
            "failure_rate",
            ascending=False
        )

    def hourly_summary(self):
        data = self.df.copy()

        data["attempt_time"] = pd.to_datetime(
            data["attempt_time"]
        )

        data["hour"] = (
            data["attempt_time"]
            .dt.hour
        )

        result = (
            data
            .groupby("hour")
            .agg(
                attempts=("attempt_id", "count"),
                failures=(
                    "attempt_status",
                    lambda x: (
                        x == "Failed"
                    ).sum()
                )
            )
            .reset_index()
        )

        result["failure_rate"] = (
            result["failures"]
            / result["attempts"]
            * 100
        )

        return result
'''

(ROOT / "src" / "__init__.py").write_text(
    files["__init__.py"],
    encoding="utf-8"
)

for name, content in files.items():
    (SRC / name).write_text(
        content,
        encoding="utf-8"
    )

(ROOT / "requirements.txt").write_text(
    "pandas\n"
    "pymongo\n"
    "mysql-connector-python\n"
    "python-dotenv\n"
    "streamlit\n",
    encoding="utf-8"
)

(ROOT / ".gitignore").write_text(
    ".env\n"
    "__pycache__/\n"
    ".ipynb_checkpoints/\n"
    "*.pyc\n",
    encoding="utf-8"
)

print("ROUTESENSE MODULAR BACKEND CREATED")

for file in sorted(SRC.iterdir()):
    print(file.name)

In [ ]:
import sys
from pathlib import Path

ROOT = Path(
    r"<PROJECT_ROOT>"
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.database import load_integrated_data
from src.mongodb import (
    get_collection,
    get_kpis,
    get_failure_summary,
    get_route_hotspots
)

print("MODULAR BACKEND TEST")

df = load_integrated_data()

print("MySQL module")
print("Integrated rows:", len(df))

collection = get_collection()

print("MongoDB module")
print("MongoDB documents:", collection.count_documents({}))

kpis = get_kpis(collection)

print("\nKPIs:")
print(kpis)

failure_data = get_failure_summary(collection)

print(
    "\nFailure categories:",
    len(failure_data)
)

hotspot_data = get_route_hotspots(collection)

print(
    "Hotspot routes:",
    len(hotspot_data)
)

print("\nMODULAR BACKEND TEST PASSED")

In [ ]:
from pathlib import Path

ROOT = Path(
    r"<PROJECT_ROOT>"
)

APP = ROOT / "app.py"

APP.write_text(
r'''
import streamlit as st
import pandas as pd

from src.database import load_integrated_data
from src.mongodb import (
    get_collection,
    get_kpis,
    get_failure_summary,
    get_hourly_summary,
    get_route_hotspots
)


st.set_page_config(
    page_title="RouteSense",
    page_icon="🚚",
    layout="wide",
    initial_sidebar_state="expanded"
)


@st.cache_data
def load_sql_data():
    return load_integrated_data()


@st.cache_resource
def load_mongo_collection():
    return get_collection()


def format_number(value):
    return f"{int(value):,}"


def main():

    st.title("🚚 RouteSense")
    st.caption(
        "Last-Mile Delivery Intelligence & Route Risk Analysis"
    )

    try:

        df = load_sql_data()
        collection = load_mongo_collection()
        kpis = get_kpis(collection)

        st.sidebar.title("RouteSense")
        st.sidebar.caption(
            "Delivery Intelligence Platform"
        )

        page = st.sidebar.radio(
            "Navigation",
            [
                "Overview",
                "Failure Analysis",
                "Route Hotspots",
                "Route Explorer",
                "System Status"
            ]
        )

        st.sidebar.divider()

        st.sidebar.success("MySQL ● Connected")
        st.sidebar.success("MongoDB ● Connected")

        if page == "Overview":

            st.header("Delivery Overview")

            c1, c2, c3, c4 = st.columns(4)

            c1.metric(
                "Total Attempts",
                format_number(
                    kpis["total_attempts"]
                )
            )

            c2.metric(
                "Failed Attempts",
                format_number(
                    kpis["failed_attempts"]
                )
            )

            c3.metric(
                "Delivered Attempts",
                format_number(
                    kpis["delivered_attempts"]
                )
            )

            c4.metric(
                "Failure Rate",
                f"{kpis['failure_rate']:.2f}%"
            )

            st.divider()


            st.subheader("Data Integration Status")

            sql_records = len(df)

            mongo_records = collection.count_documents(
                {}
            )

            unique_events = len(
                collection.distinct("event_id")
            )

            duplicate_events = (
                df["attempt_id"]
                .duplicated()
                .sum()
            )

            q1, q2, q3, q4 = st.columns(4)

            q1.metric(
                "SQL Records",
                format_number(sql_records)
            )

            q2.metric(
                "MongoDB Events",
                format_number(mongo_records)
            )

            q3.metric(
                "Unique Events",
                format_number(unique_events)
            )

            q4.metric(
                "Duplicate Event IDs",
                format_number(duplicate_events)
            )

            if (
                sql_records == mongo_records
                and mongo_records == unique_events
                and duplicate_events == 0
            ):
                st.success(
                    "✅ SQL → Python → MongoDB integration "
                    "validated successfully."
                )
            else:
                st.warning(
                    "⚠️ Data integration validation "
                    "requires attention."
                )

            st.divider()

            st.subheader("Failure Overview")

            failure_results = get_failure_summary(
                collection
            )

            failure_df = pd.DataFrame(
                failure_results
            )

            if not failure_df.empty:

                failure_df = failure_df.rename(
                    columns={
                        "_id": "Failure Reason",
                        "failure_count": "Failures",
                        "average_delay":
                            "Average Delay (min)"
                    }
                )

                chart_df = (
                    failure_df[
                        [
                            "Failure Reason",
                            "Failures"
                        ]
                    ]
                    .set_index("Failure Reason")
                )

                st.bar_chart(chart_df)

                st.dataframe(
                    failure_df,
                    use_container_width=True,
                    hide_index=True
                )

            st.divider()

            st.subheader("Hourly Failure Rate")

            hourly_results = get_hourly_summary(
                collection
            )

            hourly_df = pd.DataFrame(
                hourly_results
            )

            if not hourly_df.empty:

                hourly_df["Hour"] = (
                    hourly_df["_id"].astype(int)
                )

                hourly_df["Hour Label"] = (
                    hourly_df["Hour"]
                    .apply(
                        lambda x:
                        f"{int(x):02d}:00"
                    )
                )

                hourly_chart = (
                    hourly_df[
                        [
                            "Hour",
                            "failure_rate"
                        ]
                    ]
                    .set_index("Hour")
                    .rename(
                        columns={
                            "failure_rate":
                            "Failure Rate (%)"
                        }
                    )
                )

                st.line_chart(
                    hourly_chart,
                    use_container_width=True
                )

                st.caption(
                    "Failure rate = failed attempts ÷ "
                    "total attempts for each hour."
                )

                top_hours = (
                    hourly_df[
                        [
                            "Hour Label",
                            "total_attempts",
                            "failed_attempts",
                            "failure_rate"
                        ]
                    ]
                    .sort_values(
                        "failure_rate",
                        ascending=False
                    )
                    .head(5)
                    .rename(
                        columns={
                            "Hour Label": "Hour",
                            "total_attempts": "Attempts",
                            "failed_attempts": "Failures",
                            "failure_rate":
                                "Failure Rate (%)"
                        }
                    )
                )

                top_hours["Failure Rate (%)"] = (
                    top_hours["Failure Rate (%)"]
                    .round(2)
                )

                st.write("Highest observed hourly failure rates")
                st.dataframe(
                    top_hours,
                    use_container_width=True,
                    hide_index=True
                )

        elif page == "Failure Analysis":

            st.header("Failure Analysis")

            failure_results = get_failure_summary(
                collection
            )

            failure_df = pd.DataFrame(
                failure_results
            )

            if failure_df.empty:

                st.warning(
                    "No failure records found."
                )

            else:

                failure_df = failure_df.rename(
                    columns={
                        "_id": "Failure Reason",
                        "failure_count": "Failures",
                        "average_delay":
                            "Average Delay (min)"
                    }
                )

                c1, c2 = st.columns(2)

                c1.metric(
                    "Failure Categories",
                    len(failure_df)
                )

                c2.metric(
                    "Failed Attempts",
                    format_number(
                        failure_df["Failures"].sum()
                    )
                )

                st.divider()

                st.bar_chart(
                    failure_df
                    .set_index("Failure Reason")
                    ["Failures"]
                )

                failure_df[
                    "Average Delay (min)"
                ] = failure_df[
                    "Average Delay (min)"
                ].round(2)

                st.dataframe(
                    failure_df,
                    use_container_width=True,
                    hide_index=True
                )

                st.download_button(
                    "Download Failure Analysis",
                    failure_df.to_csv(
                        index=False
                    ).encode("utf-8"),
                    "failure_analysis.csv",
                    "text/csv"
                )

        elif page == "Route Hotspots":

            st.header("Route Hotspots")

            st.caption(
                "Route scores combine failure rate, "
                "repeat-failure rate and order volume."
            )

            hotspot_results = get_route_hotspots(
                collection
            )

            hotspot_df = pd.DataFrame(
                hotspot_results
            )

            if hotspot_df.empty:

                st.warning(
                    "No hotspot data available."
                )

            else:

                min_orders = st.slider(
                    "Minimum Orders",
                    min_value=100,
                    max_value=5000,
                    value=100,
                    step=100
                )

                filtered = hotspot_df[
                    hotspot_df["total_orders"]
                    >= min_orders
                ].copy()

                filtered = filtered.rename(
                    columns={
                        "_id": "Route",
                        "total_orders": "Orders",
                        "failed_orders":
                            "Failed Orders",
                        "repeat_failure_orders":
                            "Repeat Failure Orders",
                        "failure_rate":
                            "Failure Rate (%)",
                        "repeat_failure_rate":
                            "Repeat Failure Rate (%)",
                        "hotspot_score":
                            "Hotspot Score"
                    }
                )

                for column in [
                    "Failure Rate (%)",
                    "Repeat Failure Rate (%)",
                    "Hotspot Score"
                ]:
                    filtered[column] = (
                        filtered[column]
                        .round(2)
                    )

                st.metric(
                    "Routes Displayed",
                    len(filtered)
                )

                st.dataframe(
                    filtered,
                    use_container_width=True,
                    hide_index=True
                )

                st.download_button(
                    "Download Route Hotspots",
                    filtered.to_csv(
                        index=False
                    ).encode("utf-8"),
                    "route_hotspots.csv",
                    "text/csv"
                )

        elif page == "Route Explorer":

            st.header("Route Explorer")

            routes = sorted(
                df["route_id"]
                .dropna()
                .unique()
                .tolist()
            )

            if not routes:

                st.warning(
                    "No route data available."
                )

            else:

                selected_route = st.selectbox(
                    "Select Route",
                    routes
                )

                route_data = df[
                    df["route_id"]
                    == selected_route
                ]

                total_orders = (
                    route_data["order_id"]
                    .nunique()
                )

                failed_orders = (
                    route_data[
                        route_data["attempt_status"]
                        == "Failed"
                    ]["order_id"]
                    .nunique()
                )

                failure_rate = (
                    failed_orders
                    / total_orders
                    * 100
                    if total_orders
                    else 0
                )

                metadata = route_data.iloc[0]

            

                hotspot_results = get_route_hotspots(
                    collection
                )

                route_hotspot = next(
                    (
                        item
                        for item in hotspot_results
                        if item["_id"]
                        == selected_route
                    ),
                    None
                )

                st.subheader(
                    f"{selected_route} — "
                    f"{metadata['route_name']}"
                )

                r1, r2, r3, r4 = st.columns(4)

                r1.metric(
                    "Orders",
                    format_number(total_orders)
                )

                r2.metric(
                    "Failed Orders",
                    format_number(failed_orders)
                )

                r3.metric(
                    "Failure Rate",
                    f"{failure_rate:.2f}%"
                )

                if route_hotspot:

                    r4.metric(
                        "Hotspot Score",
                        f"{route_hotspot['hotspot_score']:.2f}"
                    )

                else:

                    r4.metric(
                        "Hotspot Score",
                        "N/A"
                    )

                if route_hotspot:

                    st.divider()

                    h1, h2 = st.columns(2)

                    h1.metric(
                        "Repeat Failure Rate",
                        f"{route_hotspot['repeat_failure_rate']:.2f}%"
                    )

                    h2.metric(
                        "Repeat Failure Orders",
                        format_number(
                            route_hotspot[
                                "repeat_failure_orders"
                            ]
                        )
                    )

                st.divider()

                info1, info2, info3, info4 = (
                    st.columns(4)
                )

                info1.write(
                    f"**Zone:** "
                    f"{metadata['order_zone']}"
                )

                info2.write(
                    f"**Origin Hub:** "
                    f"{metadata['origin_hub']}"
                )

                info3.write(
                    f"**Driver:** "
                    f"{metadata['driver_name']}"
                )

                info4.write(
                    f"**Vehicle:** "
                    f"{metadata['vehicle_type']}"
                )

                st.divider()

                st.subheader(
                    "Failure Reasons"
                )

                reasons = (
                    route_data[
                        route_data["attempt_status"]
                        == "Failed"
                    ]["failure_reason"]
                    .value_counts()
                )

                if reasons.empty:

                    st.info(
                        "No failures recorded "
                        "for this route."
                    )

                else:

                    st.bar_chart(
                        reasons
                    )

        elif page == "System Status":

            st.header("System Status")

            st.subheader(
                "Database Connectivity"
            )

            c1, c2 = st.columns(2)

            with c1:

                st.success(
                    "● MySQL — Connected"
                )

                st.write(
                    "Database: `delivery_db`"
                )

            with c2:

                st.success(
                    "● MongoDB — Connected"
                )

                st.write(
                    "Database: `routesense`"
                )

                st.write(
                    "Collection: "
                    "`delivery_events`"
                )

            st.divider()

            st.subheader(
                "Dataset Integrity"
            )

            sql_rows = len(df)

            mongo_rows = collection.count_documents(
                {}
            )

            unique_events = len(
                collection.distinct(
                    "event_id"
                )
            )

            checks = {
                "SQL integrated rows":
                    sql_rows == 117916,

                "MongoDB event documents":
                    mongo_rows == 117916,

                "Unique event IDs":
                    unique_events == 117916,

                "No missing order IDs":
                    df["order_id"]
                    .isna()
                    .sum() == 0,

                "No missing route IDs":
                    df["route_id"]
                    .isna()
                    .sum() == 0,

                "No duplicate attempt IDs":
                    df["attempt_id"]
                    .duplicated()
                    .sum() == 0
            }

            for name, passed in checks.items():

                if passed:
                    st.success(
                        f"✅ {name}"
                    )
                else:
                    st.error(
                        f"❌ {name}"
                    )

            st.divider()

            st.subheader(
                "Dataset Summary"
            )

            s1, s2, s3 = st.columns(3)

            s1.metric(
                "SQL Records",
                f"{sql_rows:,}"
            )

            s2.metric(
                "MongoDB Records",
                f"{mongo_rows:,}"
            )

            s3.metric(
                "Unique Events",
                f"{unique_events:,}"
            )

    except Exception as e:

        st.error(
            "RouteSense could not load the "
            "application data."
        )

        st.warning(
            "Check that MySQL and MongoDB are "
            "running and that the .env password "
            "is configured correctly."
        )

        with st.expander(
            "Technical error details"
        ):
            st.code(str(e))


if __name__ == "__main__":
    main()
''',
    encoding="utf-8"
)

print("ROUTESENSE UI")

print(APP)

In [ ]:
import os
from pathlib import Path
from pymongo import MongoClient
from dotenv import load_dotenv

ROOT = Path(
    r"<PROJECT_ROOT>"
)

load_dotenv(ROOT / ".env", override=True)

atlas_uri = os.getenv("ROUTESENSE_ATLAS_URI")
atlas_user = os.getenv("ROUTESENSE_ATLAS_USER")
atlas_password = os.getenv("ROUTESENSE_ATLAS_PASSWORD")

print("URI loaded:", bool(atlas_uri))
print("User loaded:", bool(atlas_user))
print("Password loaded:", bool(atlas_password))

if not atlas_uri:
    raise RuntimeError("Atlas URI missing")

if not atlas_user:
    raise RuntimeError("Atlas username missing")

if not atlas_password:
    raise RuntimeError("Atlas password missing")

client = MongoClient(
    atlas_uri,
    username=atlas_user,
    password=atlas_password,
    authSource="admin",
    serverSelectionTimeoutMS=10000
)

client.admin.command("ping")

db = client["routesense"]
collection = db["delivery_events"]

print("ATLAS CONNECTION SUCCESSFUL")
print("Database:", db.name)
print("Collection:", collection.name)
print(
    "Documents currently in Atlas:",
    collection.count_documents({})
)

client.close()

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient

ROOT = Path(
    r"<PROJECT_ROOT>"
)

load_dotenv(ROOT / ".env", override=True)

atlas_uri = os.getenv("ROUTESENSE_ATLAS_URI")
atlas_user = os.getenv("ROUTESENSE_ATLAS_USER")
atlas_password = os.getenv("ROUTESENSE_ATLAS_PASSWORD")

local_client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

atlas_client = MongoClient(
    atlas_uri,
    username=atlas_user,
    password=atlas_password,
    authSource="admin",
    serverSelectionTimeoutMS=10000
)

local_collection = (
    local_client["routesense"]["delivery_events"]
)

atlas_collection = (
    atlas_client["routesense"]["delivery_events"]
)

local_count = local_collection.count_documents({})
atlas_before = atlas_collection.count_documents({})

print("Local documents:", local_count)
print("Atlas documents before:", atlas_before)

if local_count != 117916:
    raise RuntimeError(
        f"Expected 117916 local documents, found {local_count}"
    )

if atlas_before != 0:
    raise RuntimeError(
        "Atlas collection is not empty. Migration stopped."
    )

batch = []
batch_size = 2000
inserted = 0

for document in local_collection.find({}):
    document.pop("_id", None)
    batch.append(document)

    if len(batch) == batch_size:
        atlas_collection.insert_many(
            batch,
            ordered=False
        )
        inserted += len(batch)
        batch.clear()

        print(
            f"Transferred {inserted:,} / "
            f"{local_count:,}"
        )

if batch:
    atlas_collection.insert_many(
        batch,
        ordered=False
    )
    inserted += len(batch)

atlas_after = atlas_collection.count_documents({})

print()
print("MIGRATION COMPLETE")
print("Transferred:", inserted)
print("Atlas documents:", atlas_after)

local_client.close()
atlas_client.close()
